# Auto-fit: `da4ml.pack.fit_model` places the N=64 model on a $60 `xcau7p`

Runs the predictive fitter end to end: three synth-only calibration anchors,
arithmetic search over the predicate front, then one confirming place-and-route
of the winner (`bram(8,7)`, zero DSP) at 3.0 ns. Produces `fit_front.csv` and
the predicted-vs-measured numbers in the thesis validation table.

**To run:** edit the config cell below — `VIVADO_SETTINGS`, `CHECKPOINT`, and the
dataset paths. Three synth-only jobs plus one route; well under an hour.

In [1]:
import os
os.environ.setdefault('KERAS_BACKEND', 'jax')
os.environ.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION', '0.5')

import sys
from pathlib import Path

CHECKPOINT = Path('../../../models/jsc_plf/64-16/'
                  'epoch=4433-val_acc=0.817-ebops=40646-val_loss=0.534.keras')
PARTICLES, FEATURES = 64, 16

PART_NAME       = 'xcau7p-sbvc484-2-e'
BUDGET          = dict(lut=37440, ff=74880, bram=108, dsp=216)   # Vivado-confirmed
VIVADO_SETTINGS = '/tools/Xilinx/2025.1/Vivado/settings64.sh'
CLOCK_PERIOD, LATENCY_CUTOFF = 4.0, 3.0
TRACE_BATCH     = 2048

HGQ_JSC150 = Path('..').resolve()  # shared helpers: model.py, data.py, dump_bram.tcl
DATA_DIR   = Path('../../../dataset/jsc_plf').resolve()  # from prepare_datasets.sh
NB_DIR     = Path('.').resolve()
if str(HGQ_JSC150) not in sys.path:
    sys.path.insert(0, str(HGQ_JSC150))

print(f'Target : {PART_NAME}  {BUDGET}')
print(f'Clock  : {CLOCK_PERIOD} ns / {LATENCY_CUTOFF} ns')


Target : xcau7p-sbvc484-2-e  {'lut': 37440, 'ff': 74880, 'bram': 108, 'dsp': 216}
Clock  : 4.0 ns / 3.0 ns


Load and trace the model into a `CombLogic`.

In [2]:
import keras
from model import SameDim0
from data import get_data
from hgq.utils import trace_minmax
from da4ml.converter import trace_model
from da4ml.trace import HWConfig, comb_trace

raw_ckpt = CHECKPOINT.expanduser().resolve()
traced   = raw_ckpt.parent / f'model_traced_n{PARTICLES}_f{FEATURES}.keras'
if not traced.exists():
    print('Running trace_minmax (first run only)...')
    m0 = keras.models.load_model(raw_ckpt, compile=False, custom_objects={'SameDim0': SameDim0})
    (Xtr, _), (Xv, _), _ = get_data(DATA_DIR, PARTICLES, FEATURES == 3)
    trace_minmax(m0, Xtr, batch_size=TRACE_BATCH, reset=True,  verbose=True)
    trace_minmax(m0, Xv,  batch_size=TRACE_BATCH, reset=False, verbose=True)
    m0.save(traced)
model = keras.models.load_model(traced, compile=False, custom_objects={'SameDim0': SameDim0})

inp, out = trace_model(model, hwconf=HWConfig(1, -1, -1), solver_options={'hard_dc': 2}, verbose=False)
comb = comb_trace(inp, out)
print(f'Traced N=64: comb.cost={comb.cost:.0f}, lookups={sum(1 for op in comb.ops if op.opcode==8)}')


Traced N=64: comb.cost=46065, lookups=3819


The headline call: calibrate, predict, select, confirm.

In [3]:
from da4ml.pack import fit_model

result = fit_model(
    comb,                                   # your traced CombLogic
    budget=BUDGET,
    part_name=PART_NAME,
    vivado_settings=VIVADO_SETTINGS,
    clock_period=CLOCK_PERIOD, latency_cutoff=LATENCY_CUTOFF,
    objective=('min_dsp', 'min_bram'),
    confirm=True,                           # optional single route to validate
    work_dir=str(NB_DIR / '_fit_work'),
    verbose=True,
)
print()
print(result.summary())


[vivado] start A0_baseline
[vivado] start A1_bram_max
[vivado] start A2_dsp_cal
[vivado] done A0_baseline  rc=0  t=230s
[vivado] done A1_bram_max  rc=0  t=239s
[vivado] done A2_dsp_cal  rc=0  t=240s
[synth] A0_baseline    LUT*=36321 FF=19631 BRAM=0.0 DSP=0
[synth] A1_bram_max    LUT*=30599 FF=33390 BRAM=350.0 DSP=0
[synth] A2_dsp_cal     LUT*=34704 FF=17587 BRAM=0.0 DSP=214
ResourceModel(part='xcau7p-sbvc484-2-e', clk=4.0, cutoff=3.0
  LUT*  = 18028 + 0.3971*cost - 0.3581*dsp_adder_cost  (route x0.977)
  FF*   = 12487 + 0.2828*reg_bits  (route x0.999)
  BRAM  = exact RAMB18/RAMB36 tiles from IR geometry
  DSP   = 0.2685 * n_promoted_adders)
[route] jsc  LUT=34724 FF=33379 BRAM=24.0 DSP=0  WNS=+0.050 Fmax=253.2MHz  t=345s
FIT: bram(8, 8)_dsp-
  predicted (route): LUT 33670 (90%)  FF 27008 (36%)  BRAM 24.5 (23%)  DSP 0 (0%)  latency 21 cyc
  actual  (1 core): LUT 34724 (+3.1%)  FF 33379 (+23.6%)  BRAM 24.0  DSP 0  Fmax 253.2 MHz

FIT: bram(8, 8)_dsp-
  predicted (route): LUT 33670 (90%) 

Inspect the result: chosen predicate, predicted front, feasibility.

In [4]:
import pandas as pd

print('feasible :', result.feasible)
if not result.feasible:
    print('binding  :', result.binding, '  (these exceed the margined budget)')
else:
    print('chosen   :', result.config.name)
    print('predicted:', result.predicted)
    print('total    :', result.total, f'(x{result.n_instances})')
    if result.confirm:
        print('confirmed:', result.confirm)

# Full predicted front, sorted by LUT, with a fits/fails flag against the budget.
rows = []
for cand in result.front:
    p = cand.predicted
    rows.append(dict(config=cand.config.name,
                     lut=p.lut, ff=p.ff, bram=p.bram, dsp=p.dsp, latency=p.latency,
                     fits=result.budget.fits(p * result.n_instances)))
front_df = pd.DataFrame(rows).sort_values('lut').reset_index(drop=True)
front_df.to_csv(NB_DIR / 'fit_front.csv', index=False)
print(f'\nPredicted front ({len(front_df)} candidates; {int(front_df["fits"].sum())} fit):')
fmt = {'lut': '{:.0f}'.format, 'ff': '{:.0f}'.format, 'bram': '{:.1f}'.format,
       'dsp': '{:.0f}'.format, 'latency': '{:.0f}'.format}
print(front_df.head(20).to_string(index=False, formatters=fmt))

# Budget context.
u = result.budget.usable()
print(f'\nBudget {BUDGET}')
print(f'Usable after margins: LUT {u.lut:.0f}  FF {u.ff:.0f}  BRAM {u.bram:.1f}  DSP {u.dsp:.0f}')


feasible : True
chosen   : bram(8, 8)_dsp-
predicted: ResourceVector(lut=33669.82040482323, ff=27007.654053524075, bram=24.5, dsp=0.0, latency=21)
total    : ResourceVector(lut=33669.82040482323, ff=27007.654053524075, bram=24.5, dsp=0.0, latency=21) (x1)
confirmed: {'lut': 34724.0, 'ff': 33379.0, 'bram': 24.0, 'dsp': 0.0, 'wns': 0.05, 'fmax': 253.1645569620253}

Predicted front (77 candidates; 21 fit):
          config   lut    ff  bram dsp latency  fits
 bram(4, 4)_dsp- 29895 33357 350.5   0      34 False
 bram(5, 4)_dsp- 30013 32984 303.0   0      32 False
 bram(6, 4)_dsp- 30305 30910 213.5   0      29 False
 bram(4, 5)_dsp- 30646 30512 215.5   0      30 False
 bram(5, 5)_dsp- 30712 29653 189.0   0      27 False
 bram(6, 5)_dsp- 30867 28683 151.5   0      26 False
bram(8, 6)_dsp11 30963 27224  45.0 214      23 False
 bram(7, 4)_dsp- 30998 29570 123.0   0      27 False
 bram(7, 5)_dsp- 31349 27902  99.0   0      25  True
 bram(4, 6)_dsp- 31457 29574 137.0   0      29 False
 bram(5, 6

Open the confirm route's full post-route reports.

In [5]:
import re, shlex, subprocess, time, shutil
from da4ml.codegen import RTLModel

WORK     = NB_DIR / '_fit_work'
REPORTS  = WORK / 'jsc' / 'output_jsc' / 'reports'
PR_UTIL  = REPORTS / 'jsc_post_route_util.rpt'
PR_TIM   = REPORTS / 'jsc_post_route_timing.rpt'

def _used(label, text):
    m = re.search(r'^\|\s*' + re.escape(label) + r'\s*\|\s*([\d.]+)\s*\|', text, re.MULTILINE)
    return float(m.group(1)) if m else float('nan')

if result.config is None:
    print('No feasible config was chosen — nothing to route. Binding resource(s):', result.binding)
else:
    cfg = result.config
    print(f'Chosen predicate : {cfg.name}')
    print(f'  bram_thr={cfg.bram_thr}  dsp_bo={cfg.dsp_bo}  dsp_clocked={cfg.dsp_clocked}')
    print(f'  predicted latency: {result.predicted.latency} cycles')

    # Reuse the confirm route if present; otherwise build + route the chosen config now.
    if not PR_UTIL.exists():
        print('\nNo confirm route found — applying the chosen MarkConfig and routing now (slow)...')
        marked = cfg.apply(comb, LATENCY_CUTOFF)
        out = WORK / 'jsc'
        if out.exists(): shutil.rmtree(out)
        out.mkdir(parents=True)
        RTLModel(marked, 'jsc', str(out), latency_cutoff=LATENCY_CUTOFF, clock_period=CLOCK_PERIOD,
                 part_name=PART_NAME, clock_uncertainty=0.0).write()
        tcl = out / 'build_vivado_prj.tcl'
        t0 = time.time()
        subprocess.run(['bash', '-lc',
                        f'source {shlex.quote(VIVADO_SETTINGS)} && '
                        f'vivado -mode batch -source {shlex.quote(str(tcl.resolve()))} -nojournal -nolog'],
                       cwd=str(out), capture_output=True, text=True)
        print(f'  route done  t={time.time()-t0:.0f}s')

    text = PR_UTIL.read_text() if PR_UTIL.exists() else ''
    rows = [('CLB LUTs', BUDGET['lut']), ('CLB Registers', BUDGET['ff']),
            ('Block RAM Tile', BUDGET['bram']), ('RAMB18', None), ('RAMB36/FIFO*', None),
            ('DSPs', BUDGET['dsp'])]
    print(f'\nPost-route utilisation on {PART_NAME}:')
    print(f'  {"resource":<16s} {"used":>9s} {"avail":>8s} {"util%":>7s}  verdict')
    print(f'  {"-"*16} {"-"*9} {"-"*8} {"-"*7}  {"-"*7}')
    for label, avail in rows:
        u = _used(label, text)
        if u != u:
            continue
        if avail is not None:
            pct = u / avail * 100
            print(f'  {label:<16s} {u:>9.1f} {avail:>8d} {pct:>6.1f}%  {"FIT" if u <= avail else "OVER"}')
        else:
            print(f'  {label:<16s} {u:>9.1f} {"":>8s} {"":>7s}  (primitive)')

    wns = float("nan")
    if PR_TIM.exists():
        m = re.search(r'WNS\(ns\)[^\n]*\n[^\n]*\n\s*([-\d.]+)', PR_TIM.read_text())
        wns = float(m.group(1)) if m else float("nan")
    fmax = 1000.0 / (CLOCK_PERIOD - wns) if wns == wns and (CLOCK_PERIOD - wns) > 0 else float("nan")
    print(f'\n  WNS = {wns:+.3f} ns   Fmax = {fmax:.1f} MHz   '
          f'(target {CLOCK_PERIOD} ns / {1000/CLOCK_PERIOD:.0f} MHz)')

    res_ok = all(_used(l, text) <= a for l, a in
                 [('CLB LUTs', BUDGET['lut']), ('CLB Registers', BUDGET['ff']),
                  ('Block RAM Tile', BUDGET['bram']), ('DSPs', BUDGET['dsp'])]
                 if _used(l, text) == _used(l, text))
    print(f'\n  VERDICT: resources {"FIT" if res_ok else "DO NOT FIT"} the part; '
          f'timing {"MET" if wns >= 0 else "NOT met"} at {CLOCK_PERIOD} ns.')
    if result.confirm and result.predicted:
        pm = result.predicted
        print(f'\n  predicted vs actual: LUT {pm.lut:.0f}->{result.confirm["lut"]:.0f}  '
              f'FF {pm.ff:.0f}->{result.confirm["ff"]:.0f}  '
              f'BRAM {pm.bram:.1f}->{result.confirm["bram"]:.1f}  '
              f'DSP {pm.dsp:.0f}->{result.confirm["dsp"]:.0f}')


Chosen predicate : bram(8, 8)_dsp-
  bram_thr=(8, 8)  dsp_bo=None  dsp_clocked=False
  predicted latency: 21 cycles

Post-route utilisation on xcau7p-sbvc484-2-e:
  resource              used    avail   util%  verdict
  ---------------- --------- -------- -------  -------
  CLB LUTs           34724.0    37440   92.7%  FIT
  CLB Registers      33379.0    74880   44.6%  FIT
  Block RAM Tile        24.0      108   22.2%  FIT
  RAMB18                44.0                   (primitive)
  RAMB36/FIFO*           2.0                   (primitive)
  DSPs                   0.0      216    0.0%  FIT

  WNS = +0.050 ns   Fmax = 253.2 MHz   (target 4.0 ns / 250 MHz)

  VERDICT: resources FIT the part; timing MET at 4.0 ns.

  predicted vs actual: LUT 33670->34724  FF 27008->33379  BRAM 24.5->24.0  DSP 0->0
